# Solution 1 — Ridge Regression with Engineered Features

A linear baseline + feature engineering (day-of-week dummies, log price, promo interactions, Fourier seasonality, linear trend).

Linear models are a natural first choice here: the dataset is small (730 rows), the target structure is mostly additive (trend + weekly + yearly + price + promo), and a well-engineered linear model is easy to reason about and hard to beat without much more data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

sns.set_theme(style='whitegrid')
DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])
test  = pd.read_csv(DATA_DIR / 'test.csv',  parse_dates=['date'])
train.head()

## 1. EDA

In [ ]:
print('shape:', train.shape)
print('date range:', train['date'].min(), '->', train['date'].max())
print('missing values:', train.isna().sum().sum())
train.describe().round(2)

In [ ]:
# Sales over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train['date'], train['sales'], lw=0.8)
ax.set_title('Daily sales over 2 years')
ax.set_ylabel('sales'); ax.set_xlabel('date')
plt.show()

In [ ]:
# Weekly pattern, promo effect, price relationship
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=train, x='dow', y='sales', ax=axes[0])
axes[0].set_title('Sales by day of week (0=Mon)')
sns.boxplot(data=train, x='promo', y='sales', ax=axes[1])
axes[1].set_title('Sales by promo')
axes[2].scatter(train['price'], train['sales'], s=8, alpha=0.5)
axes[2].set_xlabel('price'); axes[2].set_ylabel('sales')
axes[2].set_title('Sales vs price (curved / log)')
plt.tight_layout(); plt.show()

In [ ]:
# Promo x weekend interaction
train['weekend'] = train['dow'].isin([5,6]).astype(int)
pivot = train.groupby(['weekend','promo'])['sales'].mean().unstack().round(1)
print(pivot)

fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(data=train, x='weekend', y='sales', hue='promo', ax=ax)
ax.set_title('Promo lift is larger on weekends')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(train[['t','dow','price','promo','sales']].corr(), annot=True, cmap='coolwarm', center=0, ax=ax)
plt.title('Correlation with raw features'); plt.show()

**Takeaways from EDA**
- Strong weekly seasonality: weekend > weekday.
- Clear negative, non-linear price effect (looks log-shaped).
- Promo boosts sales, with a larger lift on weekends (interaction).
- Mild upward trend over the two years and a subtle yearly wave.

## 2. Feature engineering

In [ ]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=df.index)
    X['t'] = df['t']                                  # linear trend
    for d in range(1, 7):                             # dow dummies (drop Mon)
        X[f'dow_{d}'] = (df['dow'] == d).astype(int)
    X['log_price'] = np.log(df['price'])              # non-linear price
    is_weekend = df['dow'].isin([5,6]).astype(int)
    X['promo'] = df['promo']
    X['promo_x_weekend'] = df['promo'] * is_weekend
    angle = 2 * np.pi * df['t'] / 365.25              # yearly seasonality
    X['yr_sin'] = np.sin(angle)
    X['yr_cos'] = np.cos(angle)
    return X

X_train = make_features(train)
X_train.head()

## 3. Time-based validation
Use the last 90 days of train as validation (same horizon as test).

In [ ]:
def score(y, p):
    return {'rmse': float(np.sqrt(mean_squared_error(y, p))),
            'mape': float(mean_absolute_percentage_error(y, p))}

cutoff = len(train) - 90
tr, va = train.iloc[:cutoff], train.iloc[cutoff:]

# Baseline: raw features
raw = ['t','dow','price','promo']
base = Ridge(alpha=1.0).fit(tr[raw], tr['sales'])
pred_base = base.predict(va[raw])
print('baseline (raw)      ', score(va['sales'], pred_base))

# Engineered
eng = Ridge(alpha=1.0).fit(make_features(tr), tr['sales'])
pred_eng = eng.predict(make_features(va))
print('engineered features ', score(va['sales'], pred_eng))

In [ ]:
# Visualize validation fit
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(va['date'], va['sales'].values, label='actual', lw=1.2)
ax.plot(va['date'], pred_base, label='raw features', lw=1, alpha=0.7)
ax.plot(va['date'], pred_eng, label='engineered', lw=1.2)
ax.legend(); ax.set_title('Validation: last 90 days of train'); plt.show()

In [ ]:
# Coefficients — check the engineered model makes sense
coef = pd.Series(eng.coef_, index=make_features(tr).columns).sort_values()
fig, ax = plt.subplots(figsize=(7,4))
coef.plot.barh(ax=ax)
ax.set_title('Ridge coefficients'); plt.show()

## 4. Fit on all train, predict test

In [ ]:
final = Ridge(alpha=1.0).fit(make_features(train), train['sales'])
pred_test = final.predict(make_features(test))

out = pd.DataFrame({'date': test['date'], 'sales_pred': pred_test.round(2)})
out.to_csv('predictions_regression.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train['date'], train['sales'], lw=0.6, label='train')
ax.plot(test['date'], pred_test, lw=1.2, color='red', label='forecast')
ax.legend(); ax.set_title('Ridge forecast for next 90 days'); plt.show()
out.head()